# Logistic Regression — False Negative Safety Screening

## Purpose

This notebook evaluates whether the counterfactual feature changes in the approved synthetic legitimate emails could weaken the detection of phishing emails. The screening uses the original training data and the fixed Term Frequency-Inverse Document Frequency representation. It does not use the held-out test set to determine safety.

The investigation follows the False Positive error analysis informed by **SHapley Additive exPlanations (SHAP)** and the counterfactual-guided generation of legitimate email variants. SHAP provides the preceding explanation of model errors; the safety assessment itself is performed through controlled feature-reduction tests.

## Method

The analysis has two levels. First, individual changed features are tested on correctly detected phishing examples from five training-only validation folds. Second, the complete feature-change pattern associated with each synthetic email is tested on the same validation setting. A feature is accepted only when it induces no additional False Negative predictions. A synthetic email must pass both the individual-feature and combined-pattern tests.

The official experiment screened **34** approved variants, examined **93** unique changed features, and identified **9** variants that passed the safety criteria. These are recorded experimental checkpoints, not newly measured results until the code is executed successfully.

In [ ]:
# =========================================================================================================

# STEP 12G: COUNTERFACTUAL FN-SAFETY SCREENING

# TOKEN-LEVEL + SYNTHETIC-PATTERN STRESS TEST

# TRAINING-ONLY VALIDATION

# =========================================================================================================

#

# PHASE 3 — COUNTERFACTUAL-GUIDED SYNTHETIC AUGMENTATION

#

# PURPOSE

# -------

# Before using synthetic legitimate emails for model retraining,

# determine whether their counterfactual token changes could weaken

# phishing detection and create false negatives.

#

# TWO LEVELS OF SAFETY TEST

# -------------------------

#

# LEVEL 1 — TOKEN SAFETY

# Each changed feature is tested independently on correctly detected

# phishing validation emails.

#

# LEVEL 2 — SYNTHETIC PATTERN SAFETY

# The complete combination of changes belonging to each synthetic email

# is tested on correctly detected phishing validation emails.

#

#

# STRICT SAFETY RULE

# ------------------

#

# A token is FN-safe only if:

#

#       induced FN flips = 0

#

# A synthetic email is FN-safe only if:

#

#       all its changed tokens are individually FN-safe

#

#       AND

#

#       its complete counterfactual pattern causes 0 induced FN flips

#

#

# IMPORTANT

# ---------

#

# - No synthetic email is added to training here.

# - No augmented Logistic Regression model is trained.

# - TF-IDF vocabulary is NOT refitted.

# - Threshold remains 0.50.

# - No coefficient is manually modified.

#

# =========================================================================================================







In [ ]:
# =========================================================================================================

# 1. IMPORTS

# =========================================================================================================



import os

import json

import hashlib



import numpy as np

import pandas as pd



from scipy import sparse



from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import StratifiedKFold





print("=" * 125)

print("STEP 12G: COUNTERFACTUAL FN-SAFETY SCREENING")

print("TOKEN-LEVEL + SYNTHETIC-PATTERN STRESS TEST")

print("=" * 125)







In [ ]:
# =========================================================================================================

# 2. VERIFY REQUIRED OBJECTS

# =========================================================================================================



required_objects_12g = [



    "X_train_tfidf",

    "y_train",



    "tfidf_vectorizer",



    "synthetic_final_approved_12f",

    "synthetic_refined_change_log_12f"

]





missing_objects_12g = [



    obj



    for obj in required_objects_12g



    if obj not in globals()

]





if missing_objects_12g:



    raise RuntimeError(



        "\nRequired Phase 3 objects are missing:\n"



        f"{missing_objects_12g}\n\n"



        "Run the verified Step 12F first."

    )







In [ ]:
# =========================================================================================================

# 3. FIXED CONFIGURATION

# =========================================================================================================



RANDOM_STATE_12G = 42



THRESHOLD_12G = 0.50



GLOBAL_C_12G = 1.0



MAX_ITER_12G = 1000





EXPECTED_TRAIN_12G = 42861



EXPECTED_FEATURES_12G = 10000



EXPECTED_APPROVED_SYNTHETIC_12G = 34





MAX_ALLOWED_INDUCED_FN_FLIPS_12G = 0





PROJECT_ROOT_12G = (
    "/content/drive/MyDrive/Logistic_Regression_SHAP_Guided_Phishing_Detection"
    if os.path.isdir("/content/drive/MyDrive")
    else os.path.join(os.getcwd(), "logistic_regression", "results", "fn_safety")
)





STEP12G_FOLDER = os.path.join(

    PROJECT_ROOT_12G,

    "Step12G_Counterfactual_FN_Safety_Screening"

)





os.makedirs(

    STEP12G_FOLDER,

    exist_ok=True

)







In [ ]:
# =========================================================================================================

# 4. VERIFY ORIGINAL TRAINING ENVIRONMENT

# =========================================================================================================



y_train_12g = (



    np.asarray(

        y_train

    )



    .astype(int)



    .reshape(-1)

)





assert X_train_tfidf.shape == (

    EXPECTED_TRAIN_12G,

    EXPECTED_FEATURES_12G

)





assert len(

    y_train_12g

) == EXPECTED_TRAIN_12G





feature_names_12g = np.asarray(

    tfidf_vectorizer.get_feature_names_out()

)





assert len(

    feature_names_12g

) == EXPECTED_FEATURES_12G





locked_vocab_12g = (

    tfidf_vectorizer.vocabulary_

)





print("\nOriginal training environment verified.")



print(

    "Training matrix:",

    X_train_tfidf.shape

)



print(

    "TF-IDF features:",

    len(

        feature_names_12g

    )

)







In [ ]:
# =========================================================================================================

# 5. VERIFY APPROVED SYNTHETIC DATASET

# =========================================================================================================



approved_synthetic_12g = (

    synthetic_final_approved_12f.copy()

)





assert len(

    approved_synthetic_12g

) == EXPECTED_APPROVED_SYNTHETIC_12G





assert (

    approved_synthetic_12g[

        "label"

    ] == 0

).all()





approved_ids_12g = (



    approved_synthetic_12g[

        "Synthetic_ID"

    ]



    .astype(int)



    .tolist()

)





print(

    "\nApproved Step 12F synthetic emails:",

    len(

        approved_synthetic_12g

    )

)







In [ ]:
# =========================================================================================================

# 6. RECOVER VALIDATED CHANGE LOG

# =========================================================================================================



change_log_12g = (



    synthetic_refined_change_log_12f[



        synthetic_refined_change_log_12f[

            "Synthetic_ID"

        ].isin(

            approved_ids_12g

        )



    ]



    .copy()

)





change_log_12g = (



    change_log_12g[



        (

            change_log_12g[

                "Modified_Occurrences"

            ] > 0

        )



        &



        (

            change_log_12g[

                "Change_Satisfied"

            ] == True

        )



    ]



    .copy()

    .reset_index(drop=True)

)





assert len(

    change_log_12g

) > 0





print(

    "Validated feature-change records:",

    len(

        change_log_12g

    )

)







In [ ]:
# =========================================================================================================

# 7. RESOLVE TF-IDF INDICES

# =========================================================================================================



resolved_feature_indices_12g = []





for _, row in change_log_12g.iterrows():





    feature = str(

        row[

            "Feature"

        ]

    )





    if feature not in locked_vocab_12g:



        raise RuntimeError(

            f"Counterfactual feature '{feature}' "

            "is missing from the locked TF-IDF vocabulary."

        )





    resolved_feature_indices_12g.append(



        int(

            locked_vocab_12g[

                feature

            ]

        )

    )





change_log_12g[

    "TFIDF_Index"

] = resolved_feature_indices_12g







In [ ]:
# =========================================================================================================

# 8. BUILD TOKEN-LEVEL STRESS CONFIGURATION

# =========================================================================================================



token_stress_config_12g = (



    change_log_12g



    .groupby(

        [

            "Feature",

            "TFIDF_Index"

        ],

        as_index=False

    )



    .agg({



        "Actual_Fraction_Removed":

            "max",



        "Synthetic_ID":

            "nunique",



        "Source_FP_Number":

            "nunique"

    })



    .rename(

        columns={



            "Actual_Fraction_Removed":

                "Stress_Reduction_Fraction",



            "Synthetic_ID":

                "Synthetic_Email_Frequency",



            "Source_FP_Number":

                "Source_FP_Frequency"

        }

    )

)





print(

    "\nUnique counterfactual tokens to FN-stress-test:",

    len(

        token_stress_config_12g

    )

)







In [ ]:
# =========================================================================================================

# 9. HELPER — APPLY FEATURE REDUCTIONS

# =========================================================================================================



def apply_feature_reduction_12g(

    X,

    feature_fraction_map

):



    scale_vector = np.ones(

        X.shape[1],

        dtype=np.float64

    )





    for feature_index, fraction_removed in feature_fraction_map.items():





        fraction_removed = float(

            fraction_removed

        )





        fraction_removed = min(

            1.0,

            max(

                0.0,

                fraction_removed

            )

        )





        scale_vector[

            int(

                feature_index

            )

        ] = (

            1.0

            -

            fraction_removed

        )





    return sparse.csr_matrix(

        X.multiply(

            scale_vector

        )

    )







In [ ]:
# =========================================================================================================

# 10. FIXED 5-FOLD TRAINING-ONLY CV

# =========================================================================================================



cv_12g = StratifiedKFold(



    n_splits=5,



    shuffle=True,



    random_state=RANDOM_STATE_12G

)





fold_splits_12g = list(



    cv_12g.split(

        X_train_tfidf,

        y_train_12g

    )

)







In [ ]:
# =========================================================================================================

# 11. BUILD PHISHING SAFETY ENVIRONMENT

# =========================================================================================================



fold_environment_12g = []





print(

    "\n" + "=" * 125

)



print(

    "BUILDING TRAINING-ONLY PHISHING SAFETY ENVIRONMENT"

)



print(

    "=" * 125

)





for fold_id, (

    train_idx,

    val_idx

) in enumerate(

    fold_splits_12g,

    start=1

):





    X_fold_train = (

        X_train_tfidf[

            train_idx

        ]

    )





    X_fold_val = (

        X_train_tfidf[

            val_idx

        ]

    )





    y_fold_train = (

        y_train_12g[

            train_idx

        ]

    )





    y_fold_val = (

        y_train_12g[

            val_idx

        ]

    )





    fold_model = LogisticRegression(



        C=GLOBAL_C_12G,



        random_state=RANDOM_STATE_12G,



        max_iter=MAX_ITER_12G

    )





    fold_model.fit(

        X_fold_train,

        y_fold_train

    )





    baseline_probability = (

        fold_model.predict_proba(

            X_fold_val

        )[:, 1]

    )





    baseline_prediction = (



        baseline_probability

        >=

        THRESHOLD_12G



    ).astype(int)





    correct_phishing_mask = (



        (y_fold_val == 1)



        &



        (baseline_prediction == 1)

    )





    correct_phishing_positions = np.where(

        correct_phishing_mask

    )[0]





    X_correct_phishing = (

        X_fold_val[

            correct_phishing_positions

        ]

    )





    baseline_correct_phishing_prob = (

        baseline_probability[

            correct_phishing_positions

        ]

    )





    fold_environment_12g.append({



        "Fold":

            fold_id,



        "Model":

            fold_model,



        "X_Correct_Phishing":

            X_correct_phishing,



        "Baseline_Probabilities":

            baseline_correct_phishing_prob,



        "Correct_Phishing_Count":

            len(

                correct_phishing_positions

            )

    })





    print(

        f"Fold {fold_id}: "

        f"correct phishing available for safety stress test = "

        f"{len(correct_phishing_positions)}"

    )







In [ ]:
# =========================================================================================================

# 12. LEVEL 1 — TOKEN FN-SAFETY TEST

# =========================================================================================================



token_fold_records_12g = []





print(

    "\n" + "=" * 125

)



print(

    "LEVEL 1: TOKEN-LEVEL COUNTERFACTUAL FN-SAFETY TEST"

)



print(

    "=" * 125

)





for _, token_row in token_stress_config_12g.iterrows():





    feature = str(

        token_row[

            "Feature"

        ]

    )





    feature_index = int(

        token_row[

            "TFIDF_Index"

        ]

    )





    reduction_fraction = float(

        token_row[

            "Stress_Reduction_Fraction"

        ]

    )





    for fold_info in fold_environment_12g:





        fold_id = int(

            fold_info[

                "Fold"

            ]

        )





        fold_model = (

            fold_info[

                "Model"

            ]

        )





        X_correct_phishing = (

            fold_info[

                "X_Correct_Phishing"

            ]

        )





        baseline_probs = (

            fold_info[

                "Baseline_Probabilities"

            ]

        )





        feature_present_mask = np.asarray(



            (

                X_correct_phishing[

                    :,

                    feature_index

                ]

                >

                0

            )

            .todense()



        ).reshape(-1)





        affected_count = int(

            feature_present_mask.sum()

        )





        if affected_count == 0:





            token_fold_records_12g.append({



                "Feature":

                    feature,



                "TFIDF_Index":

                    feature_index,



                "Reduction_Fraction":

                    reduction_fraction,



                "Fold":

                    fold_id,



                "Affected_Phishing_Count":

                    0,



                "Induced_FN_Flips":

                    0,



                "Induced_FN_Rate_Affected":

                    0.0,



                "Mean_Probability_Change":

                    0.0,



                "Minimum_Probability_After":

                    np.nan

            })





            continue





        X_affected = (

            X_correct_phishing[

                feature_present_mask

            ]

        )





        baseline_affected_prob = (

            baseline_probs[

                feature_present_mask

            ]

        )





        X_modified = (

            apply_feature_reduction_12g(



                X_affected,



                {

                    feature_index:

                        reduction_fraction

                }

            )

        )





        modified_probability = (

            fold_model.predict_proba(

                X_modified

            )[:, 1]

        )





        modified_prediction = (



            modified_probability

            >=

            THRESHOLD_12G



        ).astype(int)





        induced_fn_flips = int(

            (

                modified_prediction == 0

            ).sum()

        )





        induced_fn_rate = (

            induced_fn_flips

            /

            affected_count

        )





        probability_change = (

            modified_probability

            -

            baseline_affected_prob

        )





        token_fold_records_12g.append({



            "Feature":

                feature,



            "TFIDF_Index":

                feature_index,



            "Reduction_Fraction":

                reduction_fraction,



            "Fold":

                fold_id,



            "Affected_Phishing_Count":

                affected_count,



            "Induced_FN_Flips":

                induced_fn_flips,



            "Induced_FN_Rate_Affected":

                induced_fn_rate,



            "Mean_Probability_Change":

                float(

                    probability_change.mean()

                ),



            "Minimum_Probability_After":

                float(

                    modified_probability.min()

                )

        })





token_fold_df_12g = pd.DataFrame(

    token_fold_records_12g

)







In [ ]:
# =========================================================================================================

# 13. AGGREGATE TOKEN SAFETY

# =========================================================================================================



token_safety_12g = (



    token_fold_df_12g



    .groupby(

        [

            "Feature",

            "TFIDF_Index",

            "Reduction_Fraction"

        ],

        as_index=False

    )



    .agg({



        "Affected_Phishing_Count":

            "sum",



        "Induced_FN_Flips":

            "sum",



        "Mean_Probability_Change":

            "mean",



        "Minimum_Probability_After":

            "min"

    })

)





token_safety_12g[

    "Induced_FN_Rate"

] = np.where(



    token_safety_12g[

        "Affected_Phishing_Count"

    ] > 0,



    (

        token_safety_12g[

            "Induced_FN_Flips"

        ]



        /



        token_safety_12g[

            "Affected_Phishing_Count"

        ]

    ),



    0.0

)





token_safety_12g[

    "FN_Safe_Token"

] = (



    token_safety_12g[

        "Induced_FN_Flips"

    ]



    <=



    MAX_ALLOWED_INDUCED_FN_FLIPS_12G

)





token_safety_12g = (



    token_safety_12g



    .merge(



        token_stress_config_12g[

            [

                "Feature",

                "TFIDF_Index",

                "Synthetic_Email_Frequency",

                "Source_FP_Frequency"

            ]

        ],



        on=[

            "Feature",

            "TFIDF_Index"

        ],



        how="left"

    )

)





safe_token_count_12g = int(

    token_safety_12g[

        "FN_Safe_Token"

    ].sum()

)





risky_token_count_12g = int(

    (

        token_safety_12g[

            "FN_Safe_Token"

        ] == False

    ).sum()

)





print(

    "\nToken FN-safety results:"

)



print(

    "Safe tokens:",

    safe_token_count_12g

)



print(

    "Risky tokens:",

    risky_token_count_12g

)







In [ ]:
# =========================================================================================================

# 14. BUILD SAFE / RISKY TOKEN SETS

# =========================================================================================================



safe_token_set_12g = set(



    token_safety_12g[



        token_safety_12g[

            "FN_Safe_Token"

        ] == True



    ][

        "Feature"

    ]

    .astype(str)

    .tolist()

)





risky_token_set_12g = set(



    token_safety_12g[



        token_safety_12g[

            "FN_Safe_Token"

        ] == False



    ][

        "Feature"

    ]

    .astype(str)

    .tolist()

)







In [ ]:
# =========================================================================================================

# 15. BUILD SYNTHETIC PATTERN DEFINITIONS

# =========================================================================================================



synthetic_pattern_records_12g = []





for synthetic_id in approved_ids_12g:





    sample_changes = (



        change_log_12g[



            change_log_12g[

                "Synthetic_ID"

            ] == synthetic_id



        ]



        .copy()

    )





    feature_fraction_map = {}



    changed_features = []





    for _, change_row in sample_changes.iterrows():





        feature = str(

            change_row[

                "Feature"

            ]

        )





        feature_index = int(

            change_row[

                "TFIDF_Index"

            ]

        )





        fraction = float(

            change_row[

                "Actual_Fraction_Removed"

            ]

        )





        if feature_index in feature_fraction_map:



            feature_fraction_map[

                feature_index

            ] = max(



                feature_fraction_map[

                    feature_index

                ],



                fraction

            )



        else:



            feature_fraction_map[

                feature_index

            ] = fraction





        changed_features.append(

            feature

        )





    unique_changed_features = sorted(

        set(

            changed_features

        )

    )





    risky_features_in_sample = sorted(



        set(

            unique_changed_features

        )



        &



        risky_token_set_12g

    )





    all_tokens_individually_safe = bool(

        len(

            risky_features_in_sample

        ) == 0

    )





    synthetic_pattern_records_12g.append({



        "Synthetic_ID":

            synthetic_id,



        "Feature_Fraction_Map":

            feature_fraction_map,



        "Changed_Features":

            unique_changed_features,



        "Changed_Feature_Count":

            len(

                unique_changed_features

            ),



        "All_Tokens_Individually_Safe":

            all_tokens_individually_safe,



        "Risky_Features":

            risky_features_in_sample

    })







In [ ]:
# =========================================================================================================

# 16. LEVEL 2 — COMPLETE PATTERN FN-SAFETY TEST

# =========================================================================================================



sample_fold_records_12g = []





print(

    "\n" + "=" * 125

)



print(

    "LEVEL 2: COMPLETE SYNTHETIC-PATTERN FN-SAFETY TEST"

)



print(

    "=" * 125

)





for sample_info in synthetic_pattern_records_12g:





    synthetic_id = int(

        sample_info[

            "Synthetic_ID"

        ]

    )





    feature_fraction_map = (

        sample_info[

            "Feature_Fraction_Map"

        ]

    )





    changed_feature_indices = list(

        feature_fraction_map.keys()

    )





    for fold_info in fold_environment_12g:





        fold_id = int(

            fold_info[

                "Fold"

            ]

        )





        fold_model = (

            fold_info[

                "Model"

            ]

        )





        X_correct_phishing = (

            fold_info[

                "X_Correct_Phishing"

            ]

        )





        baseline_probs = (

            fold_info[

                "Baseline_Probabilities"

            ]

        )





        if len(

            changed_feature_indices

        ) == 0:



            affected_mask = np.zeros(

                X_correct_phishing.shape[0],

                dtype=bool

            )



        else:



            affected_count_matrix = (



                X_correct_phishing[

                    :,

                    changed_feature_indices

                ]

                >

                0

            )





            affected_mask = np.asarray(



                affected_count_matrix.sum(

                    axis=1

                )



            ).reshape(-1) > 0





        affected_count = int(

            affected_mask.sum()

        )





        if affected_count == 0:





            sample_fold_records_12g.append({



                "Synthetic_ID":

                    synthetic_id,



                "Fold":

                    fold_id,



                "Affected_Phishing_Count":

                    0,



                "Induced_FN_Flips":

                    0,



                "Induced_FN_Rate":

                    0.0,



                "Mean_Probability_Change":

                    0.0,



                "Minimum_Probability_After":

                    np.nan

            })





            continue





        X_affected = (

            X_correct_phishing[

                affected_mask

            ]

        )





        baseline_affected_probs = (

            baseline_probs[

                affected_mask

            ]

        )





        X_modified = (

            apply_feature_reduction_12g(



                X_affected,



                feature_fraction_map

            )

        )





        modified_probability = (

            fold_model.predict_proba(

                X_modified

            )[:, 1]

        )





        modified_prediction = (



            modified_probability

            >=

            THRESHOLD_12G



        ).astype(int)





        induced_flips = int(

            (

                modified_prediction == 0

            ).sum()

        )





        induced_rate = (

            induced_flips

            /

            affected_count

        )





        probability_change = (

            modified_probability

            -

            baseline_affected_probs

        )





        sample_fold_records_12g.append({



            "Synthetic_ID":

                synthetic_id,



            "Fold":

                fold_id,



            "Affected_Phishing_Count":

                affected_count,



            "Induced_FN_Flips":

                induced_flips,



            "Induced_FN_Rate":

                induced_rate,



            "Mean_Probability_Change":

                float(

                    probability_change.mean()

                ),



            "Minimum_Probability_After":

                float(

                    modified_probability.min()

                )

        })





sample_fold_df_12g = pd.DataFrame(

    sample_fold_records_12g

)







In [ ]:
# =========================================================================================================

# 17. AGGREGATE SAMPLE SAFETY

# =========================================================================================================



sample_safety_12g = (



    sample_fold_df_12g



    .groupby(

        "Synthetic_ID",

        as_index=False

    )



    .agg({



        "Affected_Phishing_Count":

            "sum",



        "Induced_FN_Flips":

            "sum",



        "Mean_Probability_Change":

            "mean",



        "Minimum_Probability_After":

            "min"

    })

)





sample_safety_12g[

    "Induced_FN_Rate"

] = np.where(



    sample_safety_12g[

        "Affected_Phishing_Count"

    ] > 0,



    (

        sample_safety_12g[

            "Induced_FN_Flips"

        ]



        /



        sample_safety_12g[

            "Affected_Phishing_Count"

        ]

    ),



    0.0

)







In [ ]:
# =========================================================================================================

# 18. ADD SYNTHETIC PATTERN META

# =========================================================================================================



pattern_meta_12g = pd.DataFrame({



    "Synthetic_ID": [



        x[

            "Synthetic_ID"

        ]



        for x in synthetic_pattern_records_12g

    ],



    "Changed_Feature_Count": [



        x[

            "Changed_Feature_Count"

        ]



        for x in synthetic_pattern_records_12g

    ],



    "Pattern_Changed_Features": [



        " | ".join(

            x[

                "Changed_Features"

            ]

        )



        for x in synthetic_pattern_records_12g

    ],



    "All_Tokens_Individually_Safe": [



        x[

            "All_Tokens_Individually_Safe"

        ]



        for x in synthetic_pattern_records_12g

    ],



    "Risky_Features": [



        " | ".join(

            x[

                "Risky_Features"

            ]

        )



        for x in synthetic_pattern_records_12g

    ]

})





sample_safety_12g = (



    sample_safety_12g



    .merge(



        pattern_meta_12g,



        on="Synthetic_ID",



        how="left"

    )

)







In [ ]:
# =========================================================================================================

# 19. FINAL SYNTHETIC FN-SAFETY RULE

# =========================================================================================================



sample_safety_12g[

    "Combined_Pattern_FN_Safe"

] = (



    sample_safety_12g[

        "Induced_FN_Flips"

    ]



    <=



    MAX_ALLOWED_INDUCED_FN_FLIPS_12G

)





sample_safety_12g[

    "Final_FN_Safe_Synthetic"

] = (



    sample_safety_12g[

        "All_Tokens_Individually_Safe"

    ]



    &



    sample_safety_12g[

        "Combined_Pattern_FN_Safe"

    ]

)







In [ ]:
# =========================================================================================================

# 20. MERGE WITH SYNTHETIC EMAIL TABLE

# =========================================================================================================

#

# FIX:

# We intentionally avoid duplicate Changed_Features names.

#

# =========================================================================================================



screened_synthetic_12g = (



    approved_synthetic_12g



    .merge(



        sample_safety_12g,



        on="Synthetic_ID",



        how="left"

    )

)





# Use the exact feature list used in the stress test.

screened_synthetic_12g[

    "Changed_Features"

] = (

    screened_synthetic_12g[

        "Pattern_Changed_Features"

    ]

)





# Remove helper pattern column after canonical copy.

screened_synthetic_12g.drop(



    columns=[

        "Pattern_Changed_Features"

    ],



    inplace=True

)





assert len(

    screened_synthetic_12g

) == EXPECTED_APPROVED_SYNTHETIC_12G







In [ ]:
# =========================================================================================================

# 21. SAFE / RISKY SYNTHETIC SUBSETS

# =========================================================================================================



fn_safe_synthetic_12g = (



    screened_synthetic_12g[



        screened_synthetic_12g[

            "Final_FN_Safe_Synthetic"

        ] == True



    ]



    .copy()

    .reset_index(drop=True)

)





fn_risky_synthetic_12g = (



    screened_synthetic_12g[



        screened_synthetic_12g[

            "Final_FN_Safe_Synthetic"

        ] == False



    ]



    .copy()

    .reset_index(drop=True)

)





safe_samples_count_12g = len(

    fn_safe_synthetic_12g

)





risky_samples_count_12g = len(

    fn_risky_synthetic_12g

)







In [ ]:
# =========================================================================================================

# 22. DISPLAY TOKEN SAFETY SUMMARY

# =========================================================================================================



print(

    "\n" + "=" * 125

)



print(

    "COUNTERFACTUAL TOKEN FN-SAFETY SUMMARY"

)



print(

    "=" * 125

)





print(

    "\nUnique changed tokens tested:",

    len(

        token_safety_12g

    )

)





print(

    "FN-safe tokens:",

    safe_token_count_12g

)





print(

    "FN-risky tokens:",

    risky_token_count_12g

)





print(

    "\nTop risky tokens:"

)





display(



    token_safety_12g



    .sort_values(



        by=[

            "Induced_FN_Flips",

            "Induced_FN_Rate"

        ],



        ascending=[

            False,

            False

        ]

    )



    .head(25)



    .style



    .format({



        "Reduction_Fraction":

            "{:.4f}",



        "Induced_FN_Rate":

            "{:.6f}",



        "Mean_Probability_Change":

            "{:+.8f}",



        "Minimum_Probability_After":

            "{:.8f}"

    })

)







In [ ]:
# =========================================================================================================

# 23. DISPLAY SYNTHETIC SAFETY SUMMARY

# =========================================================================================================



print(

    "\n" + "=" * 125

)



print(

    "SYNTHETIC EMAIL COUNTERFACTUAL FN-SAFETY SUMMARY"

)



print(

    "=" * 125

)





print(

    "\nApproved synthetic emails entering safety screen:",

    EXPECTED_APPROVED_SYNTHETIC_12G

)





print(

    "Synthetic emails passing FN-safety screen:",

    safe_samples_count_12g

)





print(

    "Synthetic emails rejected for FN risk:",

    risky_samples_count_12g

)







In [ ]:
# =========================================================================================================

# 24. DISPLAY SYNTHETIC SAFETY TABLE

# =========================================================================================================



display(



    screened_synthetic_12g[



        [

            "Synthetic_ID",

            "Source_FP_Number",

            "Changed_Feature_Count",

            "Changed_Features",

            "All_Tokens_Individually_Safe",

            "Risky_Features",

            "Affected_Phishing_Count",

            "Induced_FN_Flips",

            "Induced_FN_Rate",

            "Mean_Probability_Change",

            "Final_FN_Safe_Synthetic"

        ]

    ]



    .sort_values(



        by=[

            "Final_FN_Safe_Synthetic",

            "Induced_FN_Flips"

        ],



        ascending=[

            False,

            True

        ]

    )



    .style



    .format({



        "Induced_FN_Rate":

            "{:.6f}",



        "Mean_Probability_Change":

            "{:+.8f}"

    })

)







In [ ]:
# =========================================================================================================

# 25. VERIFY SAFE SYNTHETIC DATASET

# =========================================================================================================



if safe_samples_count_12g > 0:





    assert (

        fn_safe_synthetic_12g[

            "label"

        ] == 0

    ).all()





    assert (

        fn_safe_synthetic_12g[

            "Final_FN_Safe_Synthetic"

        ] == True

    ).all()





    assert (

        fn_safe_synthetic_12g[

            "clean_text"

        ]

        .duplicated()

        .sum()

        == 0

    )







In [ ]:
# =========================================================================================================

# 26. BUILD TRAINING-READY SAFE SYNTHETIC DATASET

# =========================================================================================================



training_ready_fn_safe_12g = (



    fn_safe_synthetic_12g[



        [

            "Synthetic_ID",

            "Source_FP_Number",

            "clean_text",

            "label",

            "Changed_Features",

            "TFIDF_Cosine_Similarity",

            "Baseline_Phishing_Probability",

            "Induced_FN_Flips",

            "Induced_FN_Rate"

        ]

    ]



    .copy()

)







In [ ]:
# =========================================================================================================

# 27. SAVE OUTPUTS

# =========================================================================================================



TOKEN_SAFETY_PATH_12G = os.path.join(

    STEP12G_FOLDER,

    "Step12G_Counterfactual_Token_FN_Safety.csv"

)





TOKEN_FOLD_PATH_12G = os.path.join(

    STEP12G_FOLDER,

    "Step12G_Token_FN_Safety_Fold_Details.csv"

)





SAMPLE_SAFETY_PATH_12G = os.path.join(

    STEP12G_FOLDER,

    "Step12G_Synthetic_Email_FN_Safety.csv"

)





SAMPLE_FOLD_PATH_12G = os.path.join(

    STEP12G_FOLDER,

    "Step12G_Synthetic_Pattern_FN_Safety_Fold_Details.csv"

)





SAFE_SYNTHETIC_PATH_12G = os.path.join(

    STEP12G_FOLDER,

    "Step12G_FN_Safe_Synthetic_Training_Set.csv"

)





REJECTED_SYNTHETIC_PATH_12G = os.path.join(

    STEP12G_FOLDER,

    "Step12G_FN_Risky_Synthetic_Emails.csv"

)





CONFIG_PATH_12G = os.path.join(

    STEP12G_FOLDER,

    "Step12G_Config.json"

)





token_safety_12g.to_csv(

    TOKEN_SAFETY_PATH_12G,

    index=False

)





token_fold_df_12g.to_csv(

    TOKEN_FOLD_PATH_12G,

    index=False

)





screened_synthetic_12g.to_csv(

    SAMPLE_SAFETY_PATH_12G,

    index=False

)





sample_fold_df_12g.to_csv(

    SAMPLE_FOLD_PATH_12G,

    index=False

)





training_ready_fn_safe_12g.to_csv(

    SAFE_SYNTHETIC_PATH_12G,

    index=False

)





fn_risky_synthetic_12g.to_csv(

    REJECTED_SYNTHETIC_PATH_12G,

    index=False

)







In [ ]:
# =========================================================================================================

# 28. HASH SAFE DATASET

# =========================================================================================================



def sha256_file_12g(path):



    sha = hashlib.sha256()





    with open(

        path,

        "rb"

    ) as f:





        for block in iter(



            lambda:

                f.read(

                    1024 * 1024

                ),



            b""

        ):



            sha.update(

                block

            )





    return sha.hexdigest()





safe_dataset_hash_12g = (

    sha256_file_12g(

        SAFE_SYNTHETIC_PATH_12G

    )

)







In [ ]:
# =========================================================================================================

# 29. SAVE CONFIGURATION

# =========================================================================================================



config_12g = {



    "step":

        "12G",



    "phase":

        "Phase 3",



    "method":

        (

            "Counterfactual FN-safety screening of "

            "synthetic legitimate emails using token-level "

            "and combined-pattern stress testing"

        ),



    "approved_synthetic_entering_screen":

        EXPECTED_APPROVED_SYNTHETIC_12G,



    "unique_counterfactual_tokens_tested":

        int(

            len(

                token_safety_12g

            )

        ),



    "fn_safe_tokens":

        safe_token_count_12g,



    "fn_risky_tokens":

        risky_token_count_12g,



    "final_fn_safe_synthetic_emails":

        safe_samples_count_12g,



    "rejected_fn_risky_synthetic_emails":

        risky_samples_count_12g,



    "allowed_induced_FN_flips":

        MAX_ALLOWED_INDUCED_FN_FLIPS_12G,



    "cross_validation": {



        "method":

            "StratifiedKFold",



        "n_splits":

            5,



        "shuffle":

            True,



        "random_state":

            RANDOM_STATE_12G

    },



    "threshold":

        THRESHOLD_12G,



    "global_C":

        GLOBAL_C_12G,



    "safe_synthetic_dataset_sha256":

        safe_dataset_hash_12g,



    "training_augmented":

        False,



    "augmented_model_retrained":

        False,



    "tfidf_refitted":

        False,



    "threshold_tuned":

        False,



    "model_coefficients_modified":

        False

}





with open(

    CONFIG_PATH_12G,

    "w"

) as f:



    json.dump(

        config_12g,

        f,

        indent=4

    )







In [ ]:
# =========================================================================================================

# 30. PRESERVE VALIDATED SAFETY OUTPUTS

# =========================================================================================================



counterfactual_safe_tokens_12g = (



    token_safety_12g[



        token_safety_12g[

            "FN_Safe_Token"

        ] == True



    ]



    .copy()

)





counterfactual_risky_tokens_12g = (



    token_safety_12g[



        token_safety_12g[

            "FN_Safe_Token"

        ] == False



    ]



    .copy()

)





phase3_fn_safe_synthetic_12g = (

    training_ready_fn_safe_12g.copy()

)





phase3_fn_risky_synthetic_12g = (

    fn_risky_synthetic_12g.copy()

)







In [ ]:
# =========================================================================================================

# 31. FINAL CONFIRMATION

# =========================================================================================================



print(

    "\n" + "=" * 125

)



print(

    "STEP 12G COMPLETED SUCCESSFULLY"

)



print(

    "=" * 125

)





print(

    "\nApproved synthetic emails screened:",

    EXPECTED_APPROVED_SYNTHETIC_12G

)





print(

    "Counterfactual tokens tested:",

    len(

        token_safety_12g

    )

)





print(

    "FN-safe tokens:",

    safe_token_count_12g

)





print(

    "FN-risky tokens:",

    risky_token_count_12g

)





print(

    "\nFinal FN-safe synthetic emails:",

    safe_samples_count_12g

)





print(

    "Rejected FN-risky synthetic emails:",

    risky_samples_count_12g

)





print(

    "\nSafe synthetic dataset SHA-256:"

)



print(

    safe_dataset_hash_12g

)





print(

    "\nIMPORTANT:"

)



print(

    "No synthetic email was added to training in Step 12G."

)



print(

    "No augmented Logistic Regression model was trained."

)



print(

    "Locked TF-IDF vocabulary was NOT refitted."

)



print(

    "Decision threshold remained 0.50."

)



print(

    "No Logistic Regression coefficient was manually changed."

)





print(

# Safety screening is complete; no model retraining is performed in this notebook.

)





if safe_samples_count_12g > 0:



    print(

# Safety screening is complete; no model retraining is performed in this notebook.

        "identified by Step 12G for controlled training augmentation "

        "and training-only validation."

    )



else:



    print(

        "No synthetic email passed the strict FN-safety screen. "

        "Do NOT perform Phase 3 synthetic augmentation."

    )


## Results and interpretation

The recorded training-only screening identified **62 safe features** and **31 features associated with False Negative risk** among the 93 unique changes. Of the 34 approved synthetic legitimate email variants, **9 passed** both levels of screening and **25 were rejected**.

This screening provides a conservative safeguard against weakening phishing detection when introducing corrective legitimate examples. It does not, by itself, demonstrate improved performance on the held-out test set. No augmented classifier was trained here, and the original Term Frequency-Inverse Document Frequency vocabulary and decision threshold were left unchanged.